# Prelim

Notebook for ArchR preprocessing 

The results in the notebook were genreated using the 10X PBMC ATAC dataset: https://support.10xgenomics.com/single-cell-atac/datasets/1.2.0/atac_pbmc_10k_nextgem

<b>ArchR installation </b>

Install from [https://github.com/settylab/ArchR](https://github.com/settylab/ArchR)

```
library(devtools)
devtools::install_github("GreenleafLab/ArchR", ref="master", repos = BiocManager::repositories())
```

Update your ArchR with the customized version
```
R CMD INSTALL -l <PATH to R personal library> <path to Git clone >
```

Review the notebook `PBMC-RNA-standalone.ipynb` for setup instructions.
Install MACS2 for peak celling 
```
conda install -c bioconda macs2 
```



## Load data

Following files are required for using these tools:
1. ATAC fragments file. [Example](https://fh-pi-setty-m-eco-public.s3.us-west-2.amazonaws.com/single-cell-primers/scatac/atac_pbmc_10k_nextgem_fragments.tsv.gz)
2. Index for the fragments file. [Example](https://fh-pi-setty-m-eco-public.s3.us-west-2.amazonaws.com/single-cell-primers/scatac/atac_pbmc_10k_nextgem_fragments.tsv.gz.tbi)
3. Per barcode metrics. [Example](https://fh-pi-setty-m-eco-public.s3.us-west-2.amazonaws.com/single-cell-primers/scatac/atac_pbmc_10k_nextgem_singlecell.csv)

Use the above files to run ArchR using the ArchR preprocessing script: https://github.com/dpeerlab/SEACells/blob/main/notebooks/ArchR/ArchR-preprocessing-nfr-peaks.R


# Imports

In [ ]:
import os
import pandas as pd
import numpy as np

import scanpy as sc
import pyranges as pr
import warnings

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import SEACells

findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Lato'] not found. Falling back to DejaVu Sans.


# Load data

This section loads all the results from ArchR to generate an Anndata object

## ATAC

In [ ]:
data_dir = os.path.expanduser('ArchR_malignant_seacells/export/')

Load all the exported results from ArchR

### Peaks data

In [ ]:
# Peaks data
from scipy.io import mmread
counts = mmread(data_dir + 'peak_counts/counts.mtx')

In [ ]:
# Cell and peak information
cells = pd.read_csv(data_dir + 'peak_counts/cells.csv', index_col=0).iloc[:, 0]
peaks = pd.read_csv(data_dir + 'peak_counts/peaks.csv', index_col=0)
peaks.index = peaks['seqnames'] + ':' + peaks['start'].astype(str) + '-' + peaks['end'].astype(str)
peaks.head()

,seqnames,start,end,width,strand,score,replicateScoreQuantile,groupScoreQuantile,Reproducibility,GroupReplicate,nearestGene,distToGeneStart,peakType,distToTSS,nearestTSS,GC,idx,N
chr1:804471-804971,chr1,804471,804971,501,*,1.80521,0.486,0.383,2,C24._.multiome_P-1764_S-1766,FAM87B,12650,Distal,77,uc057aum.1,0.4671,1,0
chr1:817105-817605,chr1,817105,817605,501,*,23.55408,0.788,0.497,2,C4._.Rep1,FAM87B,16,Promoter,15,uc057aum.1,0.4810,2,0
chr1:817972-818472,chr1,817972,818472,501,*,2.15853,0.465,0.151,2,C15._.Rep2,FAM87B,851,Intronic,850,uc057aum.1,0.5150,3,0
chr1:820537-821037,chr1,820537,821037,501,*,5.29884,0.663,0.382,2,C14._.Other,FAM87B,3416,Distal,3415,uc057aum.1,0.5469,4,0
chr1:822093-822593,chr1,822093,822593,501,*,5.54768,0.690,0.479,2,C18._.atac_MUV35,FAM87B,4972,Distal,2794,uc057aum.1,0.5509,5,0


In [ ]:
ad = sc.AnnData(counts.T)
ad.obs_names = cells
ad.var_names = peaks.index
for col in peaks.columns:
    ad.var[col] = peaks[col]

In [ ]:
ad.X = ad.X.tocsr()

In [ ]:
ad

AnnData object with n_obs × n_vars = 83921 × 377577
    var: 'seqnames', 'start', 'end', 'width', 'strand', 'score', 'replicateScoreQuantile', 'groupScoreQuantile', 'Reproducibility', 'GroupReplicate', 'nearestGene', 'distToGeneStart', 'peakType', 'distToTSS', 'nearestTSS', 'GC', 'idx', 'N'

### SVD

In [ ]:
ad.obsm['X_svd'] = pd.read_csv(data_dir + 'svd.csv', index_col=0).loc[ad.obs_names, : ].values

### Metadata

In [ ]:
cell_meta = pd.read_csv(data_dir + 'cell_metadata.csv', index_col=0).loc[ad.obs_names, : ]
for col in cell_meta.columns:
    ad.obs[col] = cell_meta[col].values

In [ ]:
ad

AnnData object with n_obs × n_vars = 83921 × 377577
    obs: 'Sample', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'BlacklistRatio', 'Clusters', 'ReadsInPeaks', 'FRIP', 'MP'
    var: 'seqnames', 'start', 'end', 'width', 'strand', 'score', 'replicateScoreQuantile', 'groupScoreQuantile', 'Reproducibility', 'GroupReplicate', 'nearestGene', 'distToGeneStart', 'peakType', 'distToTSS', 'nearestTSS', 'GC', 'idx', 'N'
    obsm: 'X_svd'

### Gene scores

In [ ]:
# Gene scores
gene_scores = pd.read_csv(data_dir + 'gene_scores.csv', index_col=0).T

In [ ]:
ad.obsm['GeneScores'] = gene_scores.loc[ad.obs_names, :].values
ad.uns['GeneScoresColums'] = gene_scores.columns.values

In [ ]:
ad

AnnData object with n_obs × n_vars = 83921 × 377577
    obs: 'Sample', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'BlacklistRatio', 'Clusters', 'ReadsInPeaks', 'FRIP', 'MP'
    var: 'seqnames', 'start', 'end', 'width', 'strand', 'score', 'replicateScoreQuantile', 'groupScoreQuantile', 'Reproducibility', 'GroupReplicate', 'nearestGene', 'distToGeneStart', 'peakType', 'distToTSS', 'nearestTSS', 'GC', 'idx', 'N'
    uns: 'GeneScoresColums'
    obsm: 'X_svd', 'GeneScores'

# Visualizations

In [ ]:
# Leiden and UMAP
warnings.filterwarnings('ignore')
sc.pp.neighbors(ad, use_rep='X_svd')
sc.tl.umap(ad)
sc.tl.leiden(ad)
warnings.filterwarnings('default')

In [ ]:
ad

AnnData object with n_obs × n_vars = 83921 × 377577
    obs: 'Sample', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'BlacklistRatio', 'Clusters', 'ReadsInPeaks', 'FRIP', 'MP', 'leiden'
    var: 'seqnames', 'start', 'end', 'width', 'strand', 'score', 'replicateScoreQuantile', 'groupScoreQuantile', 'Reproducibility', 'GroupReplicate', 'nearestGene', 'distToGeneStart', 'peakType', 'distToTSS', 'nearestTSS', 'GC', 'idx', 'N'
    uns: 'GeneScoresColums', 'neighbors', 'umap', 'leiden', 'leiden_colors', 'MP_colors'
    obsm: 'X_svd', 'GeneScores', 'X_umap'
    obsp: 'distances', 'connectivities'

# Save

 Save the anndata object for downstream usage

In [ ]:
ad

AnnData object with n_obs × n_vars = 83921 × 377577
    obs: 'Sample', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'BlacklistRatio', 'Clusters', 'ReadsInPeaks', 'FRIP', 'MP', 'leiden'
    var: 'seqnames', 'start', 'end', 'width', 'strand', 'score', 'replicateScoreQuantile', 'groupScoreQuantile', 'Reproducibility', 'GroupReplicate', 'nearestGene', 'distToGeneStart', 'peakType', 'distToTSS', 'nearestTSS', 'GC', 'idx', 'N'
    uns: 'GeneScoresColums', 'neighbors', 'umap', 'leiden', 'leiden_colors', 'MP_colors'
    obsm: 'X_svd', 'GeneScores', 'X_umap'
    obsp: 'distances', 'connectivities'

In [ ]:
ad.obs

,Sample,TSSEnrichment,ReadsInTSS,ReadsInPromoter,ReadsInBlacklist,PromoterRatio,PassQC,NucleosomeRatio,nMultiFrags,nMonoFrags,nFrags,nDiFrags,BlacklistRatio,Clusters,ReadsInPeaks,FRIP,MP,leiden
x,,,,,,,,,,,,,,,,,,
multiome_P-6774_S-10146#CTCAGGATCTTGCTAT-1,multiome_P-6774_S-10146,1.948,2154,4422,1877,0.022375,1,1.363341,17719,41812,98816,39285,0.009497,C24,6854,0.081595,MP 5,11
multiome_P-6774_S-10146#GTAGCGCTCCACCCTG-1,multiome_P-6774_S-10146,1.479,2778,6735,2463,0.035092,1,0.712341,12519,56042,95963,27402,0.012833,C24,13316,0.118388,MP 5,2
multiome_P-6774_S-10146#GGCAATCGTCCTCCAA-1,multiome_P-6774_S-10146,2.202,2499,4807,2107,0.027021,1,1.356604,17115,37745,88950,34090,0.011844,C24,7168,0.094557,MP 2,2
multiome_P-6774_S-10146#CGGACCTAGGCCTAAT-1,multiome_P-6774_S-10146,2.146,3906,7690,1330,0.044967,1,1.099283,14671,40732,85508,30105,0.007777,C24,12262,0.149789,MP 5,2
multiome_P-6774_S-10146#CCACTTGGTTGCGGAT-1,multiome_P-6774_S-10146,1.548,1447,3414,2351,0.020204,1,1.052598,14368,41161,84487,28958,0.013913,C24,6055,0.073282,MP 2,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
atac_P-6255_S-8500#GAGTGAGTCGATGAAA-1,atac_P-6255_S-8500,9.256,2585,3256,312,0.116602,1,1.045714,2168,6825,13962,4969,0.011173,C24,4953,0.362167,MP 9,11
atac_P-6255_S-8500#GAGTGAGTCACGGGTC-1,atac_P-6255_S-8500,8.349,2264,2977,301,0.114509,1,1.098305,2133,6195,12999,4671,0.011578,C24,4575,0.368357,MP 9,11
atac_P-6255_S-8500#ACCTGCTAGGTCCTCG-1,atac_P-6255_S-8500,7.327,2220,2896,176,0.111651,1,1.750583,2332,4715,12969,5922,0.006785,C16,4072,0.430080,MP 2,11


In [ ]:
ad.write(data_dir + 'archr_for_seacells.h5ad')